# Per-prior non-centered parameterization

Hierarchical models in Bambi can use the **non-centered parameterization** for group-specific effects with random hyperpriors — rewriting `b ~ Normal(0, sigma)` as `z ~ Normal(0, 1); b = z * sigma` to improve sampling geometry.

Historically this was a single model-wide knob (`Model(..., noncentered=True)`). It is now also configurable **per `bmb.Prior`**, so the user can mix centered and non-centered parameterizations across the group-specific terms of a single model. This is particularly useful for distributional / custom-likelihood models (e.g. multi-parameter sequential sampling models) where one parameter benefits from non-centering while another does not.

In [1]:
import bambi as bmb
import numpy as np
import pandas as pd

## Setup — the sleepstudy dataset

A small hierarchical dataset where reaction time depends linearly on days of sleep deprivation, with an intercept and a slope that vary by subject.

In [2]:
data = bmb.load_data("sleepstudy")
data.head()

,Reaction,Days,Subject
0,249.5600,0,308
1,258.7047,1,308
2,250.8006,2,308
3,321.4398,3,308
4,356.8519,4,308


## The model-level knob (legacy behavior)

`Model(..., noncentered=True)` (the default) turns on the offset trick for every group-specific term that has a random `sigma` hyperprior. Inspecting the PyMC graph after `model.build()` shows the `_offset` variables this introduces.

In [3]:
def offset_vars(model):
    return sorted(v for v in model.backend.model.named_vars if v.endswith("_offset"))

m_nc = bmb.Model("Reaction ~ Days + (Days | Subject)", data, noncentered=True)
m_nc.build()
offset_vars(m_nc)

['1|Subject_offset', 'Days|Subject_offset']

Disabling at the model level removes them entirely:

In [4]:
m_c = bmb.Model("Reaction ~ Days + (Days | Subject)", data, noncentered=False)
m_c.build()
offset_vars(m_c)

[]

## New: per-prior override

`bmb.Prior` now accepts a `noncentered=` keyword. `None` (the default) inherits the model-level value; `True` / `False` overrides it for that specific prior.

Below we keep the intercept-by-subject term non-centered and force the slope-by-subject term to use the centered parameterization. We deliberately set `Model(noncentered=False)` so the inheritance path is also exercised: the intercept prior's explicit `noncentered=True` wins, and the slope prior inherits the model default.

In [5]:
hyper = lambda nc: bmb.Prior(
    "Normal",
    mu=0,
    sigma=bmb.Prior("HalfNormal", sigma=1),
    noncentered=nc,
)

priors = {
    "1|Subject": hyper(True),    # force noncentered
    "Days|Subject": hyper(False),  # force centered
}

m_mixed = bmb.Model(
    "Reaction ~ Days + (Days | Subject)",
    data,
    priors=priors,
    noncentered=False,  # the inherited model-level default does not matter here
)
m_mixed.build()
offset_vars(m_mixed)

['1|Subject_offset']

Only the intercept-by-subject term has an `_offset` companion: per-prior beats the model default in both directions.

## Distributional models — independent control across parameters

When more than one likelihood parameter has its own linear predictor, each parameter is a separate component with its own group-specific terms. The per-prior flag lets you choose the parameterization independently for each.

This is the scenario that motivates the feature in practice: in a sequential-sampling model (e.g. via HSSM, which builds on Bambi), one might want non-centered hierarchical drift but centered hierarchical threshold within a single model.

In [6]:
formula = bmb.Formula(
    "Reaction ~ 1 + (1 | Subject)",
    "sigma ~ 1 + (1 | Subject)",
)

priors = {
    "1|Subject": hyper(True),                        # parent (mu) component
    "sigma": {"1|Subject": hyper(False)},             # auxiliary component
}

m_dist = bmb.Model(formula, data, priors=priors)
m_dist.build()
offset_vars(m_dist)

['1|Subject_offset']

## Non-Normal priors with `noncentered=False`

The previous behavior was to raise `NotImplementedError` whenever a group-specific term had a non-Normal prior with a random hyperprior, even if the user only wanted the centered parameterization. With explicit `noncentered=False` on the prior, this now works — the centered branch is general.

`noncentered=True` on a non-Normal prior still raises, with an informative message naming the offending prior and pointing at the remediation.

In [7]:
st_prior = bmb.Prior(
    "StudentT",
    nu=4,
    mu=0,
    sigma=bmb.Prior("HalfNormal", sigma=1),
    noncentered=False,
)

m_st = bmb.Model(
    "Reaction ~ Days + (Days | Subject)",
    data,
    priors={"Days|Subject": st_prior},
)
m_st.build()  # used to raise NotImplementedError; now builds via the centered branch
offset_vars(m_st)

['1|Subject_offset']

## Summary

- `Model(..., noncentered=...)` remains the model-wide default.
- `bmb.Prior(..., noncentered=...)` overrides it per group-specific term.
- The combination lets users mix parameterizations within a single model — across grouping terms and across distributional components.
- `noncentered=False` is now general: any prior, including non-Normal hyperpriors, builds cleanly under the centered parameterization.